In [ ]:
import sys
from pathlib import Path

# Ensure project-root imports (e.g., utils.*) work regardless notebook launch directory.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / 'utils').is_dir():
        if str(candidate) not in sys.path:
            sys.path.append(str(candidate))
        break

import numpy as np
import seaborn as sns
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import os
from sklearn.metrics import confusion_matrix, roc_curve, auc
from utils.config import load_env_paths
from utils.mypalette import als_palette, condition_palette, cluster_palette_dict
import torch
from torchvision import transforms
#from utils.train_utils import subsample_data
from utils.start_notebook import setup_experiment

ENV_PATHS = load_env_paths(str((candidate / 'configs' / 'env.yml').resolve()))

# Set random seed for reproducibility
np.random.seed(42)

x = '240813_5'
batches = ['E1']

classifiers = ['TIClow']
test = 'ALS'
condition_palette['ALS'] = als_palette['ALS']
condition_palette['CTRL'] = als_palette['CTRL']

res_dir = ENV_PATHS['analysis_results_root_dir']
general_dir = os.path.join(res_dir, 'crosstask_classification_temp')
saving_dir = os.path.join(general_dir, 'cross_test')
model_dir = os.path.join(general_dir, 'regular')
config_dir = os.path.join(candidate, 'configs')

os.makedirs(saving_dir, exist_ok=True)

###
# LOAD EMBEDDINGS AND ANNOTATIONS, BALANCE DATASET MAKING SURE THAT CTRL UNTR SUBSETS ARE NOT OVERLAPPING
###

annotations, raw_emb = setup_experiment(x, batches)
annotations['embedding'] = list(raw_emb)

kwargs = {'fill': False, 'alpha': 1.0, 'linewidth': 2}

clustering_labels_path = os.path.join(ENV_PATHS['temp_results_root_dir'], x, 'clustering', f'best_clustering_labels_{batches[0]}.csv')

if os.path.exists(clustering_labels_path):
    clustering_labels_df = pd.read_csv(clustering_labels_path)
    if min(clustering_labels_df['cluster']) > 0:
        clustering_labels_df['cluster'] = clustering_labels_df['cluster'] - 1

    clustering_labels_df['cluster'] = clustering_labels_df['cluster'] + 1

    clustering_labels_df['simple_img_code'] = clustering_labels_df['img_code'].apply(lambda x: os.path.basename(x))
    annotations['simple_img_code'] = annotations['img_code'].apply(lambda x: os.path.basename(x))
    annotations = annotations.merge(clustering_labels_df[['simple_img_code', 'cluster']], on='simple_img_code', how='left')

    cluster_palette = cluster_palette_dict(annotations['cluster'].unique())

In [ ]:
def validate_model(model, data, label_of_interest, modality:str, batch_size=8):

    assert modality.lower() in ['image', 'embeddings'], "Modality must be either 'image' or 'embeddings'."

    pos_label = data[label_of_interest].unique()
    pos_label = pos_label[(pos_label != 'control') & (pos_label != 'untreated')][0]
    # Extract labels from the DataFrame
    labels = data[label_of_interest].map(lambda x: 1 if x == pos_label else 0).values

    if modality.lower() == 'image':
        img_paths = data['img_code'].values
        # Define the image transformation
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        ])
    else:
        embeddings = np.stack(data['embedding'].values)

    # Validation
    model_name = type(model).__name__
    if model_name != 'MLPClassifier':
        model.eval()
    y_pred = []
    all_labels = []
    with torch.no_grad():
        for i in range(0, len(labels), batch_size):
            batch_labels = labels[i:i + batch_size]
            if modality.lower() == 'image':
                batch_paths = img_paths[i:i + batch_size]
                images = [transform(Image.open(img_path).convert('RGB')) for img_path in batch_paths]
                inputs = torch.stack(images)
            else:
                inputs = torch.tensor(embeddings[i:i + batch_size], dtype=torch.float32)
            
            if model_name == 'MLPClassifier':
                probabilities = model.predict_proba(inputs)
                y_pred.extend(probabilities[:, 1])
            else:
                outputs = model(inputs)
                probabilities = torch.softmax(outputs, dim=1)
                y_pred.extend(probabilities[:, 1].cpu().numpy())
            all_labels.extend(batch_labels)

    fpr, tpr, _ = roc_curve(all_labels, y_pred)
    auc_score = auc(fpr, tpr)
    
    # Add predicted probabilities and dataset column to the original data
    data['predicted_probability'] = y_pred
    data['dataset'] = 'test (cross)'
    
    return auc_score, fpr, tpr, data

In [ ]:
for b in batches:

    model_dir_batch = os.path.join(model_dir, b)
    saving_dir_batch = os.path.join(saving_dir, b)
    os.makedirs(saving_dir_batch, exist_ok=True)

    batch_ann = annotations[annotations['experiment'] == b]

    testing_files = [file for file in os.listdir(model_dir_batch) if file.startswith(test) and file.endswith('.csv')]
    if len(testing_files) > 1:
        raise ValueError('Multiple testing files found')
    elif len(testing_files) == 1:
        testing_data = pd.read_csv(os.path.join(model_dir_batch, testing_files[0]))
    else:
        raise ValueError('No testing files found')
    
    batch_ann['short_img_code'] = batch_ann['img_code'].apply(lambda x: os.path.basename(x))
    testing_data['short_img_code'] = testing_data['img_code'].apply(lambda x: os.path.basename(x))

    testing_data = testing_data.merge(batch_ann[['short_img_code', 'embedding', 'cluster']], on='short_img_code', how='inner')
    probability_column = [col for col in testing_data.columns if 'predicted_probability' in col][0]
    testing_data.rename(columns={probability_column: 'predicted_probability'}, inplace=True)
    testing_data.drop(columns=['predicted_probability'], inplace=True)

    for c in classifiers:

        validation_results = pd.DataFrame()
        assert c != test, 'Classifier and test set must be different for cross-task evaluation'

        if 'TIC' in c:
            label_of_interest = 'mutation'
        else:
            label_of_interest = 'condition'

        results = [file for file in os.listdir(model_dir_batch) if file.startswith(c) and 'all_folds' in file and file.endswith('.csv')]

        if len(results) > 1:
            raise ValueError('Multiple results files found')
        elif len(results) == 1:
            validation_results = pd.read_csv(os.path.join(model_dir_batch, results[0]))

        validation_results['dataset'] = ['validation']*len(validation_results)
        assert set(['CTRL', c]) == set(validation_results['class'].unique().tolist()), 'Classes are not as expected'

        probability_column = [col for col in validation_results.columns if 'predicted_probability' in col][0]
        validation_results.rename(columns={probability_column: 'predicted_probability'}, inplace=True)

        if len(set(testing_data['img_code']).intersection(set(validation_results['img_code']))) > 0:
            common_images = set(testing_data['img_code']).intersection(set(validation_results['img_code']))
            common_df = validation_results[validation_results['img_code'].isin(common_images)]
            print(f'Common images dataframe length: {len(common_df)}')
            print(common_df.head(10))
            raise ValueError('Common images found')
        #validation_results = validation_results.merge(validation[['img_code', 'class']], on='img_code', how='outer')

        models = [file for file in os.listdir(model_dir_batch) if file.startswith(c) and file.endswith('.pt')]
        if len(models) > 1:
            raise ValueError('Multiple models found')
        elif len(models) == 1:
            model = torch.load(os.path.join(model_dir_batch, models[0]), weights_only=False, map_location=torch.device('cpu'))
        
        fold = models[0].split('F')[-1].split('.')[0]
        validation_results = validation_results[validation_results['fold'] == int(fold)]
        print(f'Length of validation results: {len(validation_results)}')

        auc_score, fpr, tpr, data = validate_model(model, testing_data, label_of_interest, modality='embeddings')
        data.to_csv(os.path.join(saving_dir_batch, f'MODEL_{c}_DATA_{test}_results.csv'), index=False)

        print(f'Validation AUC on {test} data: {auc_score}')

        data = pd.concat([validation_results, data], axis=0)
        data['Data - Class'] = data['dataset'] + ' - ' + data['class']
        print(data['Data - Class'].value_counts())
        positive_percentage = (data[(data['dataset'] == 'test (cross)') &\
                                    (data['predicted_probability'] > 0.5) &\
                                        (data['class']==test)].shape[0] / data[(data['dataset'] == 'test (cross)') & (data['class']==test)].shape[0]) * 100
        #print(f'Percentage of test data predicted as positive: {positive_percentage:.2f}%')

        fig, ax = plt.subplots(1, 2, figsize=(8, 5))
        positive_percentage = pd.DataFrame(columns=['Class', 'Percentage', 'Dataset'])
        for i, dataset in enumerate(['validation', 'test (cross)']):
            other_label = c if dataset == 'validation' else test
            temp_dataset = data[data['dataset'] == dataset]
            for lab in temp_dataset['class'].unique():
                value = temp_dataset[(temp_dataset['predicted_probability'] > 0.5) & (temp_dataset['class'] == lab)].shape[0] / temp_dataset[temp_dataset['class'] == lab].shape[0] * 100
                positive_percentage.loc[len(positive_percentage)] = [lab, value, dataset]
            sns.barplot(data=positive_percentage[positive_percentage['Dataset'] == dataset], x='Class', y='Percentage',
                        ax=ax[i], palette=condition_palette, alpha=0.6, order=['CTRL', other_label])
            ax[i].set_title(f'{dataset.capitalize()} - Samples predicted as {c}')
            ax[0].set_ylabel('Percentage')
            ax[i].set_xlabel('Class')
            ax[i].set_ylim(0, 100)
        plt.tight_layout()
        #plt.savefig(os.path.join(saving_dir_batch, f'MODEL_{c}_DATA_{test}_proportion.pdf'))
        plt.show()

        fig, ax = plt.subplots(2, 1, figsize=(8, 10))
        sns.kdeplot(data=data[data['dataset']=='validation'], x='predicted_probability', hue='class', ax=ax[0], palette=condition_palette, **kwargs)
        sns.kdeplot(data=data[data['dataset']=='test (cross)'], x='predicted_probability', hue='class', ax=ax[1], palette=condition_palette, **kwargs)
        ax[0].set_title('Validation')
        ax[1].set_title(f'Test (cross) - AUC = {auc_score:.2f} \n{positive_percentage[positive_percentage["Class"]==test]["Percentage"].values[0]:.2f}% of {test} predicted to be {c}')
        plt.suptitle(f'{c.replace("C", "C ")} Classifier tested on {test} data - {b}')
        plt.tight_layout()
        #plt.savefig(os.path.join(saving_dir_batch, f'MODEL_{c}_DATA_{test}_kde.pdf'))
        plt.show()

        if os.path.exists(clustering_labels_path):
            # Create a figure with a slightly larger width to accommodate the pie chart
            fig = plt.figure(figsize=(12, 10)) 
            
            # Use GridSpec to define a 2-row, 2-column grid. 
            # width_ratios=[2, 1] makes the top-left KDE plot wider than the pie chart.
            gs = gridspec.GridSpec(2, 2, width_ratios=[2, 1])
            
            # Define the axes based on the GridSpec
            ax0 = fig.add_subplot(gs[0, 0]) # Top left: original KDE plot
            ax1 = fig.add_subplot(gs[0, 1]) # Top right: Pie chart
            ax2 = fig.add_subplot(gs[1, :]) # Bottom row: spans both columns
            
            # Data preparation
            temp_data = data[data['als_label']=='ALS']
            temp_data = temp_data[temp_data['dataset']=='test (cross)']
            
            # 1. Top-Left Plot: KDE by Class
            sns.kdeplot(data=data[data['dataset']=='test (cross)'], x='predicted_probability', 
                        hue='class', ax=ax0, palette=condition_palette, **kwargs)
            ax0.set_title(f'Test (cross) - AUC = {auc_score:.2f} \n{positive_percentage[positive_percentage["Class"]==test]["Percentage"].values[0]:.2f}% of {test} predicted to be {c}')
            
            # 2. Top-Right Plot: Pie Chart for ALS Cluster Distribution
            cluster_counts = temp_data['cluster'].value_counts()
            
            # Attempt to match pie chart colors to your existing cluster_palette
            try:
                pie_colors = [cluster_palette[cluster] for cluster in cluster_counts.index]
            except (TypeError, KeyError):
                pie_colors = None # Fallback to default colors if cluster_palette isn't a dictionary
                
            ax1.pie(cluster_counts, autopct=None, 
                    colors=pie_colors, startangle=90, wedgeprops={'edgecolor': 'white'})
            ax1.set_title('ALS Cluster Distribution')

            cluster_proportions = temp_data['cluster'].value_counts(normalize=True)
    
            # Delete small clusters
            valid_clusters = cluster_proportions[cluster_proportions >= 0.10].index
            temp_data = temp_data[temp_data['cluster'].isin(valid_clusters)]
            
            # 3. Bottom Plot: KDE by Cluster
            sns.kdeplot(data=temp_data, x='predicted_probability', 
                        hue='cluster', ax=ax2, palette=cluster_palette, **kwargs)
            ax2.set_title(f'Probability distribution for {test} samples, divided by cluster')
            
            plt.suptitle(f'{c.replace("C", "C ")} Classifier tested on {test} data - {b}')
            plt.tight_layout()
            plt.savefig(os.path.join(saving_dir_batch, f'MODEL_{c}_DATA_{test}_cluster_specific.pdf'))
            plt.show()